<a href="https://colab.research.google.com/github/swati-git/GoogleCollabNotebooks/blob/main/prompting_with_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALL THE REQUISITE PACKAGES
# ==========================================
!pip install --upgrade -q openai tiktoken


In [ ]:
# ==========================================
# 2. AUTHENTICATE AND TEST THE CONNECTION
# ==========================================
import os
from google.colab import userdata
from openai import OpenAI


# Safely fetch the credential out of Colab's background Secrets Manager
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# Initialize the official SDK Client
client = OpenAI()


In [ ]:
# ==========================================
# Prompt with no lables
# ==========================================


policy_rules = """
Refund Policy:
1. Digital courses are non-refundable once >20% of videos are watched.
2. Exception: If purchased within the last 48 hours AND the user experienced a verified technical outage, issue a 50% partial refund.
3. VIP tier members bypass the 20% watch limit if they have zero prior refunds in the last 12 months.
"""


customer_case = """
Customer: Sarah (VIP Tier)
Purchase Time: 5 days ago
Course Progress: 35% watched
Issue: Wants a full refund because she changed her mind.
Prior Refund History: 1 refund granted 4 months ago.
"""


# 1. WITHOUT CoT (Forcing an immediate jump to the answer)
no_label_prompt = f"""
{policy_rules}


{customer_case}


Reply with ONLY one word: APPROVED, PARTIAL, or DENIED.
"""


res_no_label = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": no_label_prompt}],
   temperature=0.0
)



In [ ]:
# ==========================================
# Prompt with  lables
# ==========================================

cot_prompt = f"""
{policy_rules}


{customer_case}


Evaluate Sarah against the policy inside reasoning> tags.

Inside <reasoning>, write one <rule id="N"> block for EVERY policy rule,
in policy order (1, 2, 3). Include a block even if the rule clearly does
not apply.

Each <rule> block must contain exactly these four lines:
  Rule: <quote the rule word for word>
  Sarah's values: <the exact figures this rule depends on>
  Conditions: <each condition, followed by MET or NOT MET>
  Result: APPLIES or DOES NOT APPLY

After <reasoning>, write <verdict> containing exactly one word:
APPROVED, PARTIAL, or DENIED. Base it only on the Result lines above.
"""


res_label = client.chat.completions.create(
   model="gpt-4o-mini",
   messages=[{"role": "user", "content": cot_prompt}],
   temperature=0.0
)


print("--- WITHOUT lables ---")
print(res_no_cot.choices[0].message.content)


print("\n--- WITH labels ---")
print(res_label.choices[0].message.content)
